

سلول ۱ — مسیرها و خواندن فایل‌ها

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
from collections import defaultdict

PROJECT_ROOT = Path(".")

RAW_UNIPROT_DIR = PROJECT_ROOT / "Data_raw" / "uniprot"
INTERIM_LOC_DIR = PROJECT_ROOT / "Data_interim" / "localization"
NEG_DIR = PROJECT_ROOT / "Data_proc" / "negatives"
QC_DIR = PROJECT_ROOT / "Data_proc" / "qc_reports"

for d in [INTERIM_LOC_DIR, NEG_DIR, QC_DIR]:
    d.mkdir(parents=True, exist_ok=True)

UNIPROT_CC_PATH = RAW_UNIPROT_DIR / "uniprot_human_cc.tsv"

E3_NEG_PPI_PATH = NEG_DIR / "negative_e3_after_ppi_filter.csv"
DUB_NEG_PPI_PATH = NEG_DIR / "negative_dub_after_ppi_filter.csv"
ALL_NEG_PPI_PATH = NEG_DIR / "negative_all_after_ppi_filter.csv"

print("UniProt CC exists:", UNIPROT_CC_PATH.exists(), UNIPROT_CC_PATH)
print("E3 PPI-filtered neg exists:", E3_NEG_PPI_PATH.exists())
print("DUB PPI-filtered neg exists:", DUB_NEG_PPI_PATH.exists())
print("ALL PPI-filtered neg exists:", ALL_NEG_PPI_PATH.exists())

سلول ۲ — بررسی ستون‌های UniProt CC

In [ ]:
cc_head = pd.read_csv(UNIPROT_CC_PATH, sep="\t", dtype=str, nrows=5)

print("Shape head:", cc_head.shape)
print("Columns:")
print(cc_head.columns.tolist())

display(cc_head)

سلول ۳ — خواندن کامل UniProt CC و استخراج localization خام

In [ ]:
cc = pd.read_csv(
    UNIPROT_CC_PATH,
    sep="\t",
    dtype=str,
    low_memory=False,
)

print("UniProt CC raw shape:", cc.shape)
print("Columns:", cc.columns.tolist())

entry_col_candidates = ["Entry", "accession", "Accession", "UniProtKB"]
loc_col_candidates = [
    "Subcellular location [CC]",
    "cc_subcellular_location",
    "Subcellular location",
]

entry_col = next((c for c in entry_col_candidates if c in cc.columns), None)
loc_col = next((c for c in loc_col_candidates if c in cc.columns), None)

if entry_col is None:
    raise ValueError(f"Could not find UniProt accession column. Columns: {cc.columns.tolist()}")

if loc_col is None:
    raise ValueError(f"Could not find subcellular location column. Columns: {cc.columns.tolist()}")

print("Selected entry column:", entry_col)
print("Selected localization column:", loc_col)

In [ ]:
سلول ۴ — توابع استخراج و نرمال‌سازی compartment

In [ ]:
def normalize_ac(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    if x == "" or x.lower() in {"nan", "none", "null", "-", "na", "n/a"}:
        return np.nan
    x = re.split(r"[;,|]", x)[0].strip()
    x = x.split("-")[0].strip()
    return x


def clean_location_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r"SUBCELLULAR LOCATION:\s*", "", text, flags=re.I)
    text = re.sub(r"Subcellular location:\s*", "", text, flags=re.I)
    text = re.sub(r"\{.*?\}", " ", text)
    text = re.sub(r"\[.*?\]", " ", text)
    text = text.replace("Note=", " ")
    text = text.replace("Note:", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def split_compartments(text):
    """
    Split UniProt CC localization text into candidate localization phrases.
    """
    text = clean_location_text(text)
    if not text:
        return []
    
    # UniProt localization often uses ".", ";", and sometimes ",".
    parts = re.split(r"[;\n.]+", text)
    
    out = []
    for p in parts:
        p = p.strip()
        if not p:
            continue
        
        # Remove evidence-like or topology words after comma/parenthesis
        p = re.split(r"[\(\)]", p)[0].strip()
        p = re.sub(r"\s+", " ", p).strip()
        
        if not p:
            continue
        
        out.append(p.lower())
    
    # unique, order-preserving
    return list(dict.fromkeys(out))


COMPARTMENT_PATTERNS = {
    "nucleus": [
        r"\bnucleus\b",
        r"\bnucleoplasm\b",
        r"\bnucleolus\b",
        r"\bnuclear\b",
        r"\bchromosome\b",
    ],
    "cytoplasm": [
        r"\bcytoplasm\b",
        r"\bcytosol\b",
        r"\bcytoplasmic\b",
    ],
    "plasma_membrane": [
        r"\bcell membrane\b",
        r"\bplasma membrane\b",
        r"\bcell surface\b",
    ],
    "membrane": [
        r"\bmembrane\b",
    ],
    "mitochondrion": [
        r"\bmitochondrion\b",
        r"\bmitochondria\b",
        r"\bmitochondrial\b",
    ],
    "endoplasmic_reticulum": [
        r"\bendoplasmic reticulum\b",
        r"\ber membrane\b",
        r"\ber\b",
    ],
    "golgi": [
        r"\bgolgi\b",
        r"\bgolgi apparatus\b",
    ],
    "lysosome": [
        r"\blysosome\b",
        r"\blysomal\b",
    ],
    "endosome": [
        r"\bendosome\b",
        r"\bendosomal\b",
    ],
    "peroxisome": [
        r"\bperoxisome\b",
        r"\bperoxisomal\b",
    ],
    "extracellular": [
        r"\bsecreted\b",
        r"\bextracellular\b",
        r"\bextracellular space\b",
        r"\bextracellular region\b",
    ],
    "cytoskeleton": [
        r"\bcytoskeleton\b",
        r"\bactin\b",
        r"\bmicrotubule\b",
        r"\bcentrosome\b",
        r"\bspindle\b",
    ],
    "cell_junction": [
        r"\bcell junction\b",
        r"\btight junction\b",
        r"\badherens junction\b",
        r"\bdesmosome\b",
    ],
    "vesicle": [
        r"\bvesicle\b",
        r"\bsecretory vesicle\b",
    ],
    "proteasome": [
        r"\bproteasome\b",
    ],
}


def normalize_compartment_phrase(phrase):
    phrase = str(phrase).lower().strip()
    hits = []
    
    for compartment, patterns in COMPARTMENT_PATTERNS.items():
        for pat in patterns:
            if re.search(pat, phrase):
                hits.append(compartment)
                break
    
    # اگر plasma_membrane گرفتیم، membrane عمومی را حذف نکنیم؟ 
    # برای conservative filtering، membrane عمومی را نگه می‌داریم.
    return list(dict.fromkeys(hits))

سلول ۵ — ساخت localization table

In [ ]:
raw_rows = []

for ac, loc_text in zip(cc[entry_col], cc[loc_col]):
    ac_norm = normalize_ac(ac)
    if pd.isna(ac_norm):
        continue
    
    phrases = split_compartments(loc_text)
    
    for phrase in phrases:
        raw_rows.append({
            "uniprot_ac": ac_norm,
            "compartment_raw": phrase,
        })

loc_long = pd.DataFrame(raw_rows).drop_duplicates().reset_index(drop=True)

norm_rows = []

for _, row in loc_long.iterrows():
    cats = normalize_compartment_phrase(row["compartment_raw"])
    
    for cat in cats:
        norm_rows.append({
            "uniprot_ac": row["uniprot_ac"],
            "compartment_raw": row["compartment_raw"],
            "compartment": cat,
        })

loc_norm = pd.DataFrame(norm_rows).drop_duplicates().reset_index(drop=True)

print("Raw localization rows:", len(loc_long))
print("Normalized localization rows:", len(loc_norm))
print("Unique proteins with raw loc:", loc_long["uniprot_ac"].nunique() if len(loc_long) else 0)
print("Unique proteins with normalized loc:", loc_norm["uniprot_ac"].nunique() if len(loc_norm) else 0)
print("Compartments:")
print(loc_norm["compartment"].value_counts())

display(loc_long.head(10))
display(loc_norm.head(10))

loc_long.to_csv(
    INTERIM_LOC_DIR / "uniprot_localization_long.csv",
    index=False,
)

loc_norm.to_csv(
    INTERIM_LOC_DIR / "uniprot_localization_normalized.csv",
    index=False,
)

localization_parse_qc = pd.DataFrame([{
    "n_uniprot_rows": len(cc),
    "n_raw_localization_rows": len(loc_long),
    "n_normalized_localization_rows": len(loc_norm),
    "n_unique_proteins_raw_loc": loc_long["uniprot_ac"].nunique() if len(loc_long) else 0,
    "n_unique_proteins_normalized_loc": loc_norm["uniprot_ac"].nunique() if len(loc_norm) else 0,
    "n_unique_compartments": loc_norm["compartment"].nunique() if len(loc_norm) else 0,
}])

localization_parse_qc.to_csv(
    QC_DIR / "localization_parse_qc.csv",
    index=False,
)

display(localization_parse_qc)

سلول ۶ — خواندن نگاتیوهای بعد از PPI

In [ ]:
e3_neg_ppi = pd.read_csv(E3_NEG_PPI_PATH, dtype=str, low_memory=False)
dub_neg_ppi = pd.read_csv(DUB_NEG_PPI_PATH, dtype=str, low_memory=False)
negative_all_ppi = pd.read_csv(ALL_NEG_PPI_PATH, dtype=str, low_memory=False)

# label را عددی کنیم برای QC
for df in [e3_neg_ppi, dub_neg_ppi, negative_all_ppi]:
    df["label"] = df["label"].astype(int)

print("E3 after PPI:", e3_neg_ppi.shape)
print("DUB after PPI:", dub_neg_ppi.shape)
print("ALL after PPI:", negative_all_ppi.shape)

display(e3_neg_ppi.head())
display(dub_neg_ppi.head())

سلول ۷ — ساخت map localization و فیلتر کردن نگاتیوها

In [ ]:
loc_map = defaultdict(set)

for ac, comp in zip(loc_norm["uniprot_ac"], loc_norm["compartment"]):
    if pd.isna(ac) or pd.isna(comp):
        continue
    loc_map[str(ac)].add(str(comp))

print("Proteins in loc_map:", len(loc_map))


def get_loc_set(ac):
    ac = normalize_ac(ac)
    if pd.isna(ac):
        return set()
    return loc_map.get(str(ac), set())


def add_localization_info(df):
    out = df.copy()
    
    enz_locs = []
    sub_locs = []
    shared_locs = []
    has_enz_loc = []
    has_sub_loc = []
    coloc_flags = []
    
    for _, r in out.iterrows():
        enz_set = get_loc_set(r["enz_ac"])
        sub_set = get_loc_set(r["sub_ac"])
        shared = enz_set & sub_set
        
        enz_locs.append(";".join(sorted(enz_set)) if enz_set else "NA")
        sub_locs.append(";".join(sorted(sub_set)) if sub_set else "NA")
        shared_locs.append(";".join(sorted(shared)) if shared else "NA")
        has_enz_loc.append(bool(enz_set))
        has_sub_loc.append(bool(sub_set))
        coloc_flags.append(bool(shared))
    
    out["enz_compartments"] = enz_locs
    out["sub_compartments"] = sub_locs
    out["shared_compartments"] = shared_locs
    out["has_enz_loc"] = has_enz_loc
    out["has_sub_loc"] = has_sub_loc
    out["coloc_flag"] = coloc_flags
    
    return out


def filter_negative_by_localization(df, dataset_name):
    annotated = add_localization_info(df)
    
    removed = annotated[annotated["coloc_flag"]].copy()
    kept = annotated[~annotated["coloc_flag"]].copy()
    
    print("\n" + "="*80)
    print(dataset_name)
    print("Before:", len(annotated))
    print("Removed by shared localization:", len(removed))
    print("Kept:", len(kept))
    
    return kept, removed


e3_neg_after_loc, e3_removed_loc = filter_negative_by_localization(e3_neg_ppi, "E3")
dub_neg_after_loc, dub_removed_loc = filter_negative_by_localization(dub_neg_ppi, "DUB")

negative_all_after_loc = pd.concat(
    [e3_neg_after_loc, dub_neg_after_loc],
    ignore_index=True,
)

removed_all_loc = pd.concat(
    [e3_removed_loc, dub_removed_loc],
    ignore_index=True,
)

print("\nALL")
print("Before:", len(negative_all_ppi))
print("Removed:", len(removed_all_loc))
print("Kept:", len(negative_all_after_loc))

display(removed_all_loc.head())

سلول ۸ — QC فیلتر localization

In [ ]:
def qc_after_loc(before_df, after_df, removed_df, name):
    return {
        "dataset": name,
        "n_before": len(before_df),
        "n_removed_by_loc": len(removed_df),
        "n_after": len(after_df),
        "removed_fraction": len(removed_df) / len(before_df) if len(before_df) else np.nan,
        "n_after_unique_pair_id": after_df["pair_id"].nunique(),
        "n_after_duplicate_pair_id_rows": int(after_df.duplicated("pair_id").sum()),
        "n_after_missing_enzyme_class": int(after_df["enzyme_class"].isna().sum()),
        "n_after_missing_enz_ac": int(after_df["enz_ac"].isna().sum()),
        "n_after_missing_sub_ac": int(after_df["sub_ac"].isna().sum()),
        "n_after_pair_id_starts_with_nan": int(after_df["pair_id"].astype(str).str.startswith("nan|").sum()),
        "n_after_label_0": int((after_df["label"].astype(int) == 0).sum()),
        "n_removed_unique_shared_compartment_patterns": removed_df["shared_compartments"].nunique() if len(removed_df) else 0,
        "n_after_without_enz_loc": int((after_df["has_enz_loc"] == False).sum()) if "has_enz_loc" in after_df.columns else np.nan,
        "n_after_without_sub_loc": int((after_df["has_sub_loc"] == False).sum()) if "has_sub_loc" in after_df.columns else np.nan,
    }


localization_filter_qc = pd.DataFrame([
    qc_after_loc(e3_neg_ppi, e3_neg_after_loc, e3_removed_loc, "E3_negative_after_ppi_loc"),
    qc_after_loc(dub_neg_ppi, dub_neg_after_loc, dub_removed_loc, "DUB_negative_after_ppi_loc"),
    qc_after_loc(negative_all_ppi, negative_all_after_loc, removed_all_loc, "ALL_negative_after_ppi_loc"),
])

display(localization_filter_qc)

print("Label counts after localization:")
print(negative_all_after_loc["label"].value_counts(dropna=False))

print("Duplicate pair_id after localization:", negative_all_after_loc.duplicated("pair_id").sum())

print("\nRemoved shared compartments:")
display(
    removed_all_loc["shared_compartments"]
    .value_counts()
    .head(20)
)

ذخیره کن

In [ ]:
# Save localization-filtered negatives
e3_neg_after_loc.to_csv(
    NEG_DIR / "negative_e3_after_ppi_loc_filter.csv",
    index=False
)

dub_neg_after_loc.to_csv(
    NEG_DIR / "negative_dub_after_ppi_loc_filter.csv",
    index=False
)

negative_all_after_loc.to_csv(
    NEG_DIR / "negative_all_after_ppi_loc_filter.csv",
    index=False
)

# Save removed by localization
e3_removed_loc.to_csv(
    QC_DIR / "negative_e3_removed_by_loc.csv",
    index=False
)

dub_removed_loc.to_csv(
    QC_DIR / "negative_dub_removed_by_loc.csv",
    index=False
)

removed_all_loc.to_csv(
    QC_DIR / "negative_all_removed_by_loc.csv",
    index=False
)

# Save QC
localization_filter_qc.to_csv(
    QC_DIR / "localization_filter_qc.csv",
    index=False
)

print("Saved:")
print(NEG_DIR / "negative_e3_after_ppi_loc_filter.csv")
print(NEG_DIR / "negative_dub_after_ppi_loc_filter.csv")
print(NEG_DIR / "negative_all_after_ppi_loc_filter.csv")
print(QC_DIR / "negative_all_removed_by_loc.csv")
print(QC_DIR / "localization_filter_qc.csv")

In [ ]:
check_files = [
    NEG_DIR / "negative_e3_after_ppi_loc_filter.csv",
    NEG_DIR / "negative_dub_after_ppi_loc_filter.csv",
    NEG_DIR / "negative_all_after_ppi_loc_filter.csv",
    QC_DIR / "negative_e3_removed_by_loc.csv",
    QC_DIR / "negative_dub_removed_by_loc.csv",
    QC_DIR / "negative_all_removed_by_loc.csv",
    QC_DIR / "localization_filter_qc.csv",
    INTERIM_LOC_DIR / "uniprot_localization_long.csv",
    INTERIM_LOC_DIR / "uniprot_localization_normalized.csv",
    QC_DIR / "localization_parse_qc.csv",
]

for f in check_files:
    print(f.name, "exists:", f.exists())
    if f.exists() and f.suffix == ".csv":
        tmp = pd.read_csv(f, dtype=str, low_memory=False)
        print("shape:", tmp.shape)